# Ugandan Sign Language Recognition Model Training

This notebook trains a CNN model for recognizing Ugandan sign language gestures from video.

**Steps:**
1. Mount Google Drive and load dataset
2. Preprocess videos (extract frames, resize)
3. Build 3D CNN model
4. Train the model
5. Convert to TensorFlow Lite for Flutter
6. Upload to Hugging Face Space

## 1. Setup and Install Dependencies

In [ ]:
# Install required packages
!pip install -q tensorflow opencv-python-headless pandas numpy scikit-learn matplotlib seaborn tqdm

import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from tqdm import tqdm
import json

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Your dataset root
BASE_DATASET_PATH = '/content/drive/MyDrive/USL_Tutor/DATASET_ug_sign_language'

# Folder-only mode (no CSV required)
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}

def find_videos_recursive(root_dir, extensions):
    matched = []
    for current_root, _, files in os.walk(root_dir):
        for file_name in files:
            ext = os.path.splitext(file_name)[1].lower()
            if ext in extensions:
                matched.append(os.path.join(current_root, file_name))
    return matched

if os.path.exists(BASE_DATASET_PATH):
    print(f"✓ Base dataset folder found at: {BASE_DATASET_PATH}")

    VIDEO_ROOT = os.path.join(BASE_DATASET_PATH, 'videos')
    if not os.path.exists(VIDEO_ROOT):
        VIDEO_ROOT = BASE_DATASET_PATH

    print(f"✓ Video folder found at: {VIDEO_ROOT}")

    all_videos = find_videos_recursive(VIDEO_ROOT, VIDEO_EXTENSIONS)
    print(f"✓ Found {len(all_videos)} video files (recursive)")

    if all_videos:
        print("Sample video files:")
        for sample in all_videos[:5]:
            print(" -", os.path.relpath(sample, VIDEO_ROOT))
    else:
        print("✗ No videos found. Check file extensions and folder structure.")
else:
    print(f"✗ Dataset not found at: {BASE_DATASET_PATH}")
    VIDEO_ROOT = None
    all_videos = []

## 3. Configuration

In [ ]:
# Model hyperparameters (Low-RAM safe defaults for Colab)
CONFIG = {
    'IMG_SIZE': 112,
    'SEQUENCE_LENGTH': 16,
    'BATCH_SIZE': 2,              # smaller batch helps tiny datasets/generalization
    'EPOCHS': 60,                 # train longer; early stopping will stop when needed
    'LEARNING_RATE': 0.0005,
    'VALIDATION_SPLIT': 0.2,
    'TEST_SPLIT': 0.1,
    'RANDOM_SEED': 42,

    # Small-data mode
    'SMALL_DATA_MODE': True,

    # Synthetic data expansion (balanced + RAM-safe)
    'BALANCE_CLASSES': True,
    'TARGET_SAMPLES_PER_CLASS': 'auto',
    'MAX_SAMPLES_PER_CLASS': 10,   # with ~5 real samples/class, make modest expansion only
    'AUGMENT_DUPLICATES_ONLY': True,
}

# Set random seeds for reproducibility
np.random.seed(CONFIG['RANDOM_SEED'])
tf.random.set_seed(CONFIG['RANDOM_SEED'])

print("Configuration (Low-RAM mode):")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 4. Load and Explore Dataset

In [ ]:
# Build training dataframe and auto-resolve sign labels
# Output columns: video_path, sign_word

import re

if VIDEO_ROOT is None:
    raise ValueError('VIDEO_ROOT is None. Fix dataset path in section 2 first.')

all_videos = find_videos_recursive(VIDEO_ROOT, VIDEO_EXTENSIONS)
if len(all_videos) == 0:
    raise ValueError(f'No video files found under {VIDEO_ROOT}')

# Optional subset filter by top-level folder under videos/
# Examples: 'numbers', 'alphabets', 'words', or None for all
ONLY_TOP_LEVEL = 'numbers'
if ONLY_TOP_LEVEL:
    filtered_videos = []
    for vp in all_videos:
        rel = os.path.relpath(vp, VIDEO_ROOT)
        parts = rel.split(os.sep)
        if len(parts) >= 1 and parts[0].lower() == ONLY_TOP_LEVEL.lower():
            filtered_videos.append(vp)
    all_videos = filtered_videos
    print(f"Filtered to top-level '{ONLY_TOP_LEVEL}': {len(all_videos)} videos")

if len(all_videos) == 0:
    raise ValueError('No videos left after top-level filtering.')

# Label mode options:
# - 'folder_level' -> recommended for structure like videos/numbers/1/*.mp4
# - 'auto'         -> use mapping CSV if found, else filename
# - 'filename'     -> videos/numbers/0.mp4 => label '0'
LABEL_MODE = 'folder_level'

# Used when LABEL_MODE == 'folder_level'
# 1 -> videos/<label>/file.mp4
# 2 -> videos/<group>/<label>/file.mp4  (e.g., numbers/1/file.mp4 => label '1')
LABEL_FROM_LEVEL = 2

# When training numbers-only, keep only numeric labels (e.g., 0..10)
STRICT_NUMERIC_LABELS = True

# Optional mapping CSV in dataset root (ID -> sign word)
# Required columns: video_ID, sign_word
id_to_sign = {}
root_csv_files = [
    os.path.join(BASE_DATASET_PATH, f)
    for f in os.listdir(BASE_DATASET_PATH)
    if f.lower().endswith('.csv')
]

mapping_csv_used = None
for csv_path in root_csv_files:
    try:
        map_df = pd.read_csv(csv_path)
        if 'video_ID' in map_df.columns and 'sign_word' in map_df.columns:
            for _, row in map_df.iterrows():
                key = os.path.splitext(str(row['video_ID']))[0].strip().lower()
                id_to_sign[key] = str(row['sign_word']).strip().lower()
            mapping_csv_used = csv_path
            break
    except Exception:
        pass

if mapping_csv_used:
    print(f"Loaded mapping CSV with {len(id_to_sign)} IDs: {mapping_csv_used}")

records = []
for vp in all_videos:
    rel = os.path.relpath(vp, VIDEO_ROOT)
    parts = rel.split(os.sep)
    stem = os.path.splitext(os.path.basename(vp))[0].strip().lower()

    if LABEL_MODE == 'auto':
        if stem in id_to_sign:
            label = id_to_sign[stem]
        else:
            label = stem
    elif LABEL_MODE == 'filename':
        label = stem
    else:
        if len(parts) > LABEL_FROM_LEVEL:
            label = parts[LABEL_FROM_LEVEL - 1]
        elif len(parts) >= 2:
            label = parts[0]
        else:
            label = stem

    label = label.replace('_', ' ').strip().lower()
    records.append({'video_path': vp, 'sign_word': label, 'relative_path': rel})

df = pd.DataFrame(records)

if ONLY_TOP_LEVEL == 'numbers' and STRICT_NUMERIC_LABELS:
    before = len(df)
    df = df[df['sign_word'].astype(str).str.match(r'^\d+$')].copy()
    removed = before - len(df)
    print(f"Removed {removed} non-numeric samples from numbers subset")

    unique_labels = set(df['sign_word'].astype(str).unique())
    non_numeric = sorted([label for label in unique_labels if not re.match(r'^\d+$', label)])
    if non_numeric:
        raise ValueError(f"Non-numeric labels still present: {non_numeric}. Check folder structure under videos/numbers/")

print(f"Total samples from videos: {len(df)}")
print(f"LABEL_MODE = {LABEL_MODE}")
if LABEL_MODE == 'folder_level':
    print(f"LABEL_FROM_LEVEL = {LABEL_FROM_LEVEL}")

print("\nFirst few rows:")
print(df[['relative_path', 'sign_word']].head(10))

print("\nClass distribution:")
print(df['sign_word'].value_counts().head(30))

print(f"\nUnique signs: {df['sign_word'].nunique()}")

if set(df['sign_word'].unique()) <= {'alphabets', 'numbers', 'words'}:
    print("\n⚠ You are still training only 3 category labels. For sign-name prediction, use subfolders per sign (e.g., numbers/1, numbers/2, ...).")

plt.figure(figsize=(15, 6))
df['sign_word'].value_counts().head(20).plot(kind='bar')
plt.title('Top 20 Sign Words Distribution')
plt.xlabel('Sign Word')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Video Preprocessing Functions

In [ ]:
def apply_frame_augmentation(frame):
    """Apply light random augmentation to one RGB frame in [0,1]."""
    aug = frame.copy()

    if np.random.rand() < 0.5:
        aug = np.flip(aug, axis=1)

    if np.random.rand() < 0.8:
        alpha = np.random.uniform(0.9, 1.1)
        beta = np.random.uniform(-0.08, 0.08)
        aug = np.clip(alpha * aug + beta, 0.0, 1.0)

    if np.random.rand() < 0.5:
        noise = np.random.normal(0, 0.015, size=aug.shape).astype(np.float32)
        aug = np.clip(aug + noise, 0.0, 1.0)

    return aug.astype(np.float16)


def build_balanced_dataframe(df, config):
    """
    Return dataframe with optional class balancing by oversampling minority classes.
    Oversampled rows are marked with augment=True.
    """
    working = df.copy()
    working['augment'] = False

    if not config.get('BALANCE_CLASSES', False):
        return working

    class_counts = working['sign_word'].value_counts()
    current_max = int(class_counts.max())

    target_cfg = config.get('TARGET_SAMPLES_PER_CLASS', 'auto')
    if target_cfg == 'auto':
        target = current_max
    else:
        target = int(target_cfg)

    cap = int(config.get('MAX_SAMPLES_PER_CLASS', target))
    target = min(target, cap)

    balanced_parts = []
    for sign_name, group in working.groupby('sign_word'):
        group = group.copy()
        n = len(group)

        if n >= target:
            balanced_parts.append(group.sample(n=target, random_state=CONFIG['RANDOM_SEED']) if n > target else group)
            continue

        needed = target - n
        sampled = group.sample(n=needed, replace=True, random_state=CONFIG['RANDOM_SEED']).copy()
        sampled['augment'] = True if config.get('AUGMENT_DUPLICATES_ONLY', True) else False

        balanced_parts.append(group)
        balanced_parts.append(sampled)

    balanced_df = pd.concat(balanced_parts, ignore_index=True)

    print(f"Balancing target per class: {target} (current max={current_max}, cap={cap})")
    return balanced_df


def extract_frames(video_path, num_frames=30, img_size=224, augment=False):
    """
    Extract fixed number of frames from video.
    Returns numpy array of shape (num_frames, img_size, img_size, 3)
    """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return None

    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if ret:
            frame = cv2.resize(frame, (img_size, img_size))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = frame.astype(np.float16) / 255.0
            if augment:
                frame = apply_frame_augmentation(frame)
            frames.append(frame)
        else:
            if frames:
                frames.append(frames[-1])
            else:
                frames.append(np.zeros((img_size, img_size, 3), dtype=np.float16))

    cap.release()
    return np.array(frames, dtype=np.float16)


def load_video_dataset(df, config):
    """
    Load all videos and labels into memory from df columns: video_path, sign_word.
    Returns X (videos), y (labels), label_encoder
    """
    working_df = build_balanced_dataframe(df, config)

    print("Class counts before balancing:")
    print(df['sign_word'].value_counts().head(20))
    print(f"Total rows before balancing: {len(df)}")

    print("\nClass counts after balancing:")
    print(working_df['sign_word'].value_counts().head(20))
    print(f"Total rows after balancing: {len(working_df)}")

    X = []
    y = []
    failed_videos = []

    print("\nLoading videos...")
    for _, row in tqdm(working_df.iterrows(), total=len(working_df)):
        video_path = row['video_path']
        label = row['sign_word']
        use_augment = bool(row.get('augment', False))

        if not os.path.exists(video_path):
            failed_videos.append(video_path)
            continue

        frames = extract_frames(
            video_path,
            num_frames=config['SEQUENCE_LENGTH'],
            img_size=config['IMG_SIZE'],
            augment=use_augment
        )

        if frames is not None and len(frames) == config['SEQUENCE_LENGTH']:
            X.append(frames)
            y.append(label)
        else:
            failed_videos.append(video_path)

    if failed_videos:
        print(f"\nFailed to load {len(failed_videos)} videos:")
        print(failed_videos[:10])

    X = np.array(X, dtype=np.float16)
    if len(y) == 0:
        raise ValueError('No videos were successfully loaded. Check paths and video files.')

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    est_mb = X.nbytes / (1024 * 1024)
    print(f"\nLoaded {len(X)} videos successfully")
    print(f"Shape: {X.shape}")
    print(f"Classes: {len(label_encoder.classes_)}")
    print(f"Approx RAM used by X: {est_mb:.1f} MB")

    return X, y_encoded, label_encoder

print("✓ Preprocessing + balancing functions defined")

## 6. Load Data

In [ ]:
# Load the dataset
X, y, label_encoder = load_video_dataset(df, CONFIG)

# Save label encoder for later use
labels_dict = {i: label for i, label in enumerate(label_encoder.classes_)}
print("\nLabel mapping:")
print(labels_dict)

## 7. Split Dataset

In [ ]:
import math

# Robust stratified split for small datasets with many classes
n_samples = len(X)
n_classes = len(np.unique(y))

if n_samples <= n_classes:
    raise ValueError(
        f"Not enough samples ({n_samples}) for {n_classes} classes. Add more videos per class."
    )

# Ensure test set has at least one sample per class for stratification
requested_test = CONFIG['TEST_SPLIT']
min_test_count = n_classes
computed_test_count = max(int(math.ceil(requested_test * n_samples)), min_test_count)

# Keep room for training set
max_test_count = n_samples - n_classes
if max_test_count <= 0:
    raise ValueError("Dataset too small after class constraints. Add more samples.")

test_count = min(computed_test_count, max_test_count)

print(f"Samples: {n_samples}, Classes: {n_classes}")
print(f"Requested TEST_SPLIT={requested_test} -> using test_count={test_count}")

# First split: separate test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=test_count,
    random_state=CONFIG['RANDOM_SEED'],
    stratify=y
)

# Second split: validation from remaining data
n_temp = len(X_temp)
requested_val = CONFIG['VALIDATION_SPLIT']
val_count = int(math.ceil(requested_val * n_temp))

# If validation set is too small for stratification, fall back to non-stratified split
use_stratified_val = val_count >= n_classes and n_temp - val_count >= n_classes

if val_count <= 0:
    val_count = 1
if val_count >= n_temp:
    val_count = n_temp - 1

print(f"Requested VALIDATION_SPLIT={requested_val} -> using val_count={val_count}")
print(f"Validation stratified: {use_stratified_val}")

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=val_count,
    random_state=CONFIG['RANDOM_SEED'],
    stratify=y_temp if use_stratified_val else None
)

# Free temporary tensor to reduce memory pressure
del X_temp, y_temp
import gc
gc.collect()

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Convert labels to categorical
num_classes = len(label_encoder.classes_)
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

print("Split complete with adaptive safeguards.")

## 8. Build 3D CNN Model

In [ ]:
def build_3dcnn_model(input_shape, num_classes, small_data_mode=True):
    """
    Build a 3D CNN model for video classification.
    Input shape: (num_frames, height, width, channels)
    """
    if small_data_mode:
        model = keras.Sequential([
            layers.Input(shape=input_shape),

            layers.Conv3D(16, (3, 3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling3D((1, 2, 2)),
            layers.Dropout(0.2),

            layers.Conv3D(32, (3, 3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling3D((2, 2, 2)),
            layers.Dropout(0.3),

            layers.Conv3D(64, (3, 3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling3D((2, 2, 2)),
            layers.Dropout(0.3),

            layers.GlobalAveragePooling3D(),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.4),
            layers.Dense(num_classes, activation='softmax')
        ])
    else:
        model = keras.Sequential([
            layers.Input(shape=input_shape),
            layers.Conv3D(32, (3, 3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling3D((2, 2, 2)),
            layers.Dropout(0.3),

            layers.Conv3D(64, (3, 3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling3D((2, 2, 2)),
            layers.Dropout(0.3),

            layers.Conv3D(128, (3, 3, 3), activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling3D((2, 2, 2)),
            layers.Dropout(0.4),

            layers.GlobalAveragePooling3D(),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation='softmax')
        ])

    return model


# Build model
input_shape = (CONFIG['SEQUENCE_LENGTH'], CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE'], 3)
model = build_3dcnn_model(input_shape, num_classes, CONFIG.get('SMALL_DATA_MODE', True))

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE']),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

model.summary()

## 9. Training Callbacks

In [ ]:
# Define callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=6,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_usl_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.CSVLogger('training_log.csv')
]

print("✓ Callbacks configured")

## 10. Train Model

In [ ]:
# Train the model (robust to partial reruns / kernel resets)

# 1) Ensure core artifacts exist
required_globals = ['X', 'y', 'label_encoder', 'CONFIG', 'tf', 'keras', 'layers']
missing = [name for name in required_globals if name not in globals()]
if missing:
    raise RuntimeError(f"Missing prerequisites: {missing}. Run cells 3, 5, 7, 9, 11, 13 first.")

# 2) Ensure split tensors exist
if not all(name in globals() for name in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test', 'y_train_cat', 'y_val_cat', 'y_test_cat']):
    from sklearn.model_selection import train_test_split

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=CONFIG['TEST_SPLIT'],
        random_state=CONFIG['RANDOM_SEED'],
        stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=CONFIG['VALIDATION_SPLIT'],
        random_state=CONFIG['RANDOM_SEED'],
        stratify=y_temp
    )

    num_classes = len(label_encoder.classes_)
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
    y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

    print(f"Training set: {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")

# 3) Ensure model exists (rebuild if needed)
if 'model' not in globals() or model is None:
    if 'build_3dcnn_model' not in globals():
        raise RuntimeError("build_3dcnn_model is not defined. Run cell 17 first.")

    input_shape = (CONFIG['SEQUENCE_LENGTH'], CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE'], 3)
    num_classes = len(label_encoder.classes_)
    model = build_3dcnn_model(input_shape, num_classes)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE']),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
    )
    print("Rebuilt model after kernel reset.")

# 4) Ensure callbacks exist
if 'callbacks' not in globals() or callbacks is None:
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            'best_usl_model.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        keras.callbacks.CSVLogger('training_log.csv')
    ]
    print("Recreated callbacks.")

# 5) Train
history = model.fit(
    X_train, y_train_cat,
    batch_size=CONFIG['BATCH_SIZE'],
    epochs=CONFIG['EPOCHS'],
    validation_data=(X_val, y_val_cat),
    callbacks=callbacks,
    verbose=1
)
print("Training completed.")

## 11. Visualize Training Results

In [ ]:
# Plot training history (works even after kernel reset)

import os

if 'history' in globals() and history is not None:
    hist = history.history
elif os.path.exists('training_log.csv'):
    log_df = pd.read_csv('training_log.csv')
    hist = {
        'accuracy': log_df['accuracy'].tolist() if 'accuracy' in log_df.columns else [],
        'val_accuracy': log_df['val_accuracy'].tolist() if 'val_accuracy' in log_df.columns else [],
        'loss': log_df['loss'].tolist() if 'loss' in log_df.columns else [],
        'val_loss': log_df['val_loss'].tolist() if 'val_loss' in log_df.columns else [],
    }
    print("Loaded metrics from training_log.csv")
else:
    raise RuntimeError("No training history found. Run the training cell first.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
if len(hist['accuracy']) > 0:
    axes[0].plot(hist['accuracy'], label='Train Accuracy')
if len(hist['val_accuracy']) > 0:
    axes[0].plot(hist['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
if len(hist['loss']) > 0:
    axes[1].plot(hist['loss'], label='Train Loss')
if len(hist['val_loss']) > 0:
    axes[1].plot(hist['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

if len(hist['val_accuracy']) > 0:
    print(f"\nBest validation accuracy: {max(hist['val_accuracy']):.4f}")

## 12. Evaluate on Test Set

In [ ]:
# Evaluate on test set
test_loss, test_acc, test_top3_acc = model.evaluate(X_test, y_test_cat, verbose=1)

print(f"\nTest Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f}")
print(f"  Top-3 Accuracy: {test_top3_acc:.4f}")

# Get predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_classes,
    target_names=label_encoder.classes_
))

## 13. Convert to TensorFlow Lite

In [ ]:
# Convert to TensorFlow Lite (Force SELECT_TF_OPS for 3D model support)

# This model uses MaxPool3D, which is not supported by native-only TFLite.
# So we export a Flex model directly.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]

# Keep conversion simple/stable for Flex export
# (quantization can be added later after baseline works)
tflite_model = converter.convert()

tflite_path = 'usl_model_flex.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"✓ Flex TFLite model saved: {tflite_path}")
print(f"  Size: {os.path.getsize(tflite_path) / (1024*1024):.2f} MB")
print("⚠ This file requires Select TF Ops runtime on Android.")

# Save label mapping
with open('labels.json', 'w') as f:
    json.dump(labels_dict, f, indent=2)

print("✓ Labels saved: labels.json")
print(f"Model file to use: {tflite_path}")

## 14. Test TFLite Model

In [ ]:
# Load TFLite model and test
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Model Details:")
print(f"  Input shape: {input_details[0]['shape']}")
print(f"  Output shape: {output_details[0]['shape']}")

# Test on one sample
test_sample = X_test[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_sample)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details[0]['index'])

tflite_pred = np.argmax(tflite_output[0])
original_pred = np.argmax(y_pred[0])

print(f"\nTest inference:")
print(f"  Original model prediction: {label_encoder.classes_[original_pred]}")
print(f"  TFLite model prediction: {label_encoder.classes_[tflite_pred]}")
print(f"  Match: {tflite_pred == original_pred}")

## 15. Save to Google Drive

In [ ]:
# Copy models to Google Drive
output_dir = '/content/drive/MyDrive/USL_Models'
os.makedirs(output_dir, exist_ok=True)

!cp best_usl_model.h5 "{output_dir}/"
!cp usl_model_flex.tflite "{output_dir}/"
!cp labels.json "{output_dir}/"
!cp training_curves.png "{output_dir}/"
!cp training_log.csv "{output_dir}/"

print(f"✓ All files saved to: {output_dir}")
print("\nFiles:")
!ls -lh "{output_dir}"

## 16. Download for Flutter App

In [ ]:
# Download files to local machine (zip for reliability)
from google.colab import files

print("Preparing model files for download...")
!zip -j usl_model_export.zip usl_model_flex.tflite labels.json >/dev/null

files.download('usl_model_export.zip')

print("\n✓ Download complete!")
print("\nNext steps:")
print("1. Unzip usl_model_export.zip")
print("2. Copy usl_model_flex.tflite to: assets/models/")
print("3. Copy labels.json to: assets/data/")
print("4. Add to pubspec.yaml assets section")
print("5. Install tflite_flutter package")
print("6. Add Select TF Ops runtime dependency for Android")